# Pipeline d'Imagerie du Ciel Profond en Astrophotographie
*Par: Nicolas Payot, Gabriel Missael Barco, Auriane Thilloy, Olivia Pereira*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GabrielMissael/super-resolution-workshop/blob/master/notebooks/Traditional_Pipeline_DSO.ipynb)      [![View on GitHub](https://img.shields.io/badge/View_on-GitHub-black?logo=github)](https://github.com/GabrielMissael/super-resolution-workshop)

Ce notebook explore le pipeline de traitement pour l'imagerie du ciel profond (DSO - Deep-Sky Object), comme les galaxies et les nébuleuses. Contrairement à l'imagerie planétaire qui utilise des poses très courtes, le ciel profond nécessite des poses longues pour capturer des objets très peu lumineux.

Nous verrons les étapes essentielles :
1.  **Calibration** : Correction des défauts du capteur avec des images spéciales (darks, flats).
2.  **Alignement** : Recalage des images en se basant sur la position des étoiles.
3.  **Dématriçage (Debayering)** : Reconstruction des couleurs à partir des données brutes du capteur.
4.  **Empilement (Stacking)** : Combinaison des images pour améliorer le rapport signal/bruit.
5.  **Amélioration** : Réduction du bruit et amélioration des détails.

En image, voici le but de ce notebook:
<div>
<img src="https://images.nicolaspayot.ca/super-rez-workshop/pipeline_result.png" width="1200"/>
</div>


In [ ]:
# @title
# !rm -r super-resolution-workshop
!git clone --quiet https://github.com/GabrielMissael/super-resolution-workshop.git
!cd super-resolution-workshop

In [ ]:
# @title
import sys
sys.path.append('super-resolution-workshop')

from pathlib import Path
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.image as mpimg
from astropy.io import fits
from astropy.visualization import ImageNormalize, AsinhStretch
from ipywidgets import interact, FloatSlider
from IPython.display import display, clear_output

from src.process.dso_utils import normalize_flat, calibrate_frame, make_master_frame, percentile_normalize, crop, clip
from src.process.star_align import visualize_matches, compute_transforms_relative_to_reference, visualize_image_positions, warp_images_to_reference, show_match_pair
from src.demosaic.debayer import debayer_image

## 1. Chargement des Données Brutes (Images "Lights")

La première étape consiste à charger nos images brutes, aussi appelées **"lights"**. Ce sont les poses longues qui contiennent l'objet céleste que nous voulons photographier (ici, la galaxie NGC 7331).

Chaque image est affectée par des signaux non désirés (bruit thermique, poussières, etc.). L'objectif du traitement est de nettoyer ces images pour ne garder que le signal de la galaxie.

La cellule ci-dessous charge les premières images de notre séquence pour visualiser leur contenu.

In [ ]:
# @title
def frame_file(n):
    # Prefer files named with frame_NNNNN pattern; otherwise use first 4 files
    candidate = data_dir / f'frame_{n:05d}.fits'
    if candidate.exists():
        return candidate
    # fallback: use index (n-1) from sorted list if available
    idx = n - 1
    if idx < len(files):
        return files[idx]
    return None


# Ensure repo root is on path
proj_root = Path('..').resolve()
if proj_root.match("/"):
    proj_root = Path('super-resolution-workshop').resolve()
sys.path.insert(0, str(proj_root))
data_dir = proj_root / 'data' / 'ngc7331_crops'
# print('Looking for FITS in', data_dir)
if not data_dir.exists():
    raise FileNotFoundError(f'Data directory not found: {data_dir}')
else:
    files = sorted(list(data_dir.glob('*.fits')))

chosen = []
for n in [1, 2, 3, 4]:
    p = frame_file(n)
    if p is None:
        print(f'Frame {n} not found')
    else:
        chosen.append(p)
if not chosen:
    raise FileNotFoundError('No FITS files found in data directory')

# Load data once to make the slider responsive
raw_frames_data = []
for p in chosen:
    with fits.open(str(p)) as hdul:
        raw_frames_data.append(hdul[0].data.astype(float))

# Create the plot structure once
fig, axes = plt.subplots(2, 2, figsize=(12, 7))
axes = axes.ravel()
images = []
for i, (ax, p) in enumerate(zip(axes, chosen)):
    # Display initial image
    img = ax.imshow(raw_frames_data[i], cmap='gray', norm=ImageNormalize(stretch=AsinhStretch(a=0.005)))
    images.append(img)
    ax.set_title(p.name)
    ax.axis('off')

for ax in axes[len(chosen):]:
    ax.axis('off')
plt.suptitle('NGC7331 crops: frames 1-4')
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.close(fig) # Prevent double display

def update_stretch(a):
    a /= 1000
    for img in images:
        img.set_norm(ImageNormalize(stretch=AsinhStretch(a=a)))
    clear_output(wait=True)
    display(fig)

interact(update_stretch, a=FloatSlider(min=0.1, max=20, step=0.01, value=5, description='Asinh Stretch * 1000', continuous_update=False))

None

## 2. Calibration avec les Darks

Les **"darks"** sont des images prises avec le même temps de pose et la même température que les images "lights", mais avec le capuchon sur le télescope.

**Leur but ?** Capturer le **bruit thermique** du capteur. Ce bruit, qui dépend de la température et du temps de pose, ajoute des pixels lumineux parasites. En soustrayant l'image "dark" de nos images brutes, on élimine une grande partie de ce bruit indésirable.

Ici, nous chargeons une image "master dark", qui est une moyenne de plusieurs darks individuels pour réduire le bruit aléatoire qu'ils contiennent.

In [ ]:
# @title
dark_candidates = sorted((data_dir.glob('*Master*Dark*.fits')) or [])
# also try any dark*.fits if master not found
if not dark_candidates:
    dark_candidates = sorted(list(data_dir.glob('*dark*.fits')))
if not dark_candidates:
    raise FileNotFoundError('No dark frames found in data directory')

# use first dark candidate
p = dark_candidates[0]
with fits.open(str(p)) as hdul:
    arr = hdul[0].data.astype(float)
vmin, vmax = np.percentile(arr, (1, 99))
plt.figure(figsize=(8, 6))
plt.imshow(arr, cmap='gray', vmin=vmin, vmax=vmax)
plt.title('Master dark')
plt.axis('off')
plt.show()

## 3. Calibration avec les Flats

Les **"flats"** sont des images d'un fond uniformément éclairé (comme un ciel au crépuscule ou un écran spécial).

**Leur but ?** Corriger deux problèmes majeurs :
1.  **Le vignettage** : l'assombrissement des coins de l'image.
2.  **Les poussières** : les petites ombres créées par des poussières sur le capteur ou les optiques.

En divisant nos images par l'image "flat", on uniformise la luminosité et on fait disparaître ces défauts.

In [ ]:
# @title
flat_candidates = sorted(list(data_dir.glob('*Master*Flat*.fits')))
if not flat_candidates:
    flat_candidates = sorted(list(data_dir.glob('*flat*.fits')))[0:8] if list(data_dir.glob('*flat*.fits')) else []
if not flat_candidates:
    raise FileNotFoundError('No flat frames found in data directory')

p = flat_candidates[0]
with fits.open(str(p)) as hdul:
    arr = hdul[0].data.astype(float)
vmin, vmax = np.percentile(arr, (1, 99))
plt.figure(figsize=(8, 6))
plt.imshow(arr, cmap='gray', vmin=vmin, vmax=vmax)
plt.title('Flat')
plt.axis('off')
plt.show()

## 4. Processus de Calibration

Maintenant que nous avons nos images "lights", "darks" et "flats", nous pouvons calibrer chaque image brute.

La formule de calibration est la suivante : **Image Calibrée = (Image Brute - Master Dark) / (Master Flat Normalisé)**

<br>
<br>
<div>
<img src="https://images.nicolaspayot.ca/super-rez-workshop/pipeline_calibrated.png" width="1200"/>
</div>
Figure : Schéma du processus de calibration
<br>
<br>

- **Normaliser le flat** : On divise chaque pixel du master flat par la valeur médiane de l'image pour que sa valeur moyenne soit proche de 1. Cela évite de modifier la luminosité globale de l'image brute.
- **Appliquer la calibration** : On applique la formule à chaque image "light" de notre séquence.

La cellule suivante définit les fonctions nécessaires et applique ce processus à nos images.

In [ ]:
# @title
num_files = 6

with fits.open(str(flat_candidates[0])) as hdul:
    fits_data = hdul[0].data.astype(float)
    flat_norm = normalize_flat(fits_data)

with fits.open(str(dark_candidates[0])) as hdul:
    master_dark = hdul[0].data.astype(float) / 2 ** 16
calibrated_frames = []
for n in range(1, num_files + 1):
    p = frame_file(n)
    with fits.open(str(p)) as hdul:
        raw = hdul[0].data.astype(float) / 2 ** 16
    calibrated = calibrate_frame(raw, master_dark, flat_norm)
    calibrated_frames.append(calibrated)

# Load the raw frame for comparison
with fits.open(str(frame_file(2))) as hdul:
    raw = hdul[0].data.astype(float) / 2 ** 16

# Create a figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Define a common normalization for visualization
norm = ImageNormalize(stretch=AsinhStretch(a=0.01))
cmap = "gray"

# 1. Raw Frame
im1 = axes[0].imshow(percentile_normalize(raw, 20, 99.9), norm=norm, cmap=cmap)
axes[0].set_title('Raw Frame 2')
axes[0].axis('off')
# fig.colorbar(im1, ax=axes[0], shrink=0.8)

# 2. Calibrated Frame
im2 = axes[1].imshow(percentile_normalize(calibrated_frames[1], 20, 99.9), norm=norm, cmap=cmap)
axes[1].set_title('Calibrated Frame 2')
axes[1].axis('off')
# fig.colorbar(im2, ax=axes[1], shrink=0.8)

# 3. Difference (Calibrated - Raw)
difference = calibrated_frames[1] - raw
im3 = axes[2].imshow(percentile_normalize(difference, 20, 99.9), norm=norm, cmap=cmap)
axes[2].set_title('Difference (Calibrated - Raw)')
axes[2].axis('off')
# fig.colorbar(im3, ax=axes[2], shrink=0.8)

plt.suptitle('Comparison of Raw vs. Calibrated Frame')
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

## 5. Alignement par les Étoiles

À cause de la rotation de la Terre et des petites imperfections de la monture du télescope, chaque image est légèrement décalée par rapport aux autres. Avant de les empiler, il faut les aligner parfaitement.

Pour cela, on utilise les étoiles comme points de référence :
1.  **Détection d'étoiles** : Un algorithme détecte les étoiles les plus brillantes dans chaque image.
2.  **Choix d'une image de référence** : On choisit l'image la plus nette de la série comme base pour l'alignement.
3.  **Calcul des transformations** : Pour chaque autre image, l'algorithme compare la position de ses étoiles à celles de l'image de référence. Il calcule ensuite la transformation géométrique (translation, rotation) nécessaire pour superposer parfaitement les étoiles.

La cellule ci-dessous détecte les étoiles et calcule ces transformations.

In [ ]:
max_stars = 100 # @param {"type":"integer"}
min_distance = 6 # @param {"type":"integer"}
threshold_sigma = 5.0 # @param {"type":"number"}

detect_kwargs = dict(threshold_sigma=threshold_sigma, min_distance=min_distance, max_stars=max_stars)
match_kwargs = dict(max_match_dist=12.0, ransac_thresh=3.0)

res = compute_transforms_relative_to_reference(calibrated_frames, ref_idx=0,
                                               detect_kwargs=detect_kwargs,
                                               match_kwargs=match_kwargs)

transforms = res['transforms']
star_coords = res['star_coords']
matches = res['matches']

# Plot reference frame with detected stars
norm = ImageNormalize(stretch=AsinhStretch(a=0.01))
plt.figure(figsize=(8, 8))
plt.imshow(percentile_normalize(calibrated_frames[0], 20, 99.9), cmap='gray', norm=norm)
if len(star_coords[0]) > 0:
    plt.scatter(star_coords[0][:, 0], star_coords[0][:, 1], s=30, edgecolor='yellow', facecolor='none')
plt.title('Reference frame (detected stars)')
plt.axis('off')
plt.show()

### 5.1 Visualisation de l'Orientation des Images

Cette visualisation montre comment chaque image est positionnée par rapport à l'image de référence. Les décalages (translation et rotation) sont souvent très faibles, de l'ordre de quelques pixels.

Pour mieux voir ces décalages, la visualisation **exagère les transformations par un facteur 100**. Cela permet de voir clairement dans quelle direction chaque image a bougé.

- Les **carrés colorés** représentent la position originale des images.

In [ ]:
# @title
def exaggerate_transforms(transforms, trans_scale=1000.0, rot_scale=100.0):
    """Return a list of transforms where translations are multiplied by trans_scale and
    rotation/scale deviations from identity are multiplied by rot_scale.
    This exaggerates tiny shears/rotations as well as translations.
    """
    out = []
    for M in transforms:
        if M is None:
            out.append(None)
            continue
        a, b, tx = float(M[0, 0]), float(M[0, 1]), float(M[0, 2])
        c, d, ty = float(M[1, 0]), float(M[1, 1]), float(M[1, 2])
        # deviations from identity
        a_new = 1.0 + (a - 1.0) * float(rot_scale)
        d_new = 1.0 + (d - 1.0) * float(rot_scale)
        b_new = b * float(rot_scale)
        c_new = c * float(rot_scale)
        tx_new = tx * float(trans_scale)
        ty_new = ty * float(trans_scale)
        out.append(np.array([[a_new, b_new, tx_new], [c_new, d_new, ty_new]]))
    return out


shapes = [calibrated_frames[i].shape for i in range(len(calibrated_frames))]
labels = [f'Img {i}' if i != 0 else "Reference" for i in range(len(calibrated_frames))]

translation_scale = 100 # @param {"type":"integer"}
rotation_scale = 100 # @param {"type":"integer"}

if translation_scale <=0:
      raise ValueError("Translation scale must be positive")
if rotation_scale <= 0:
      raise ValueError("Rotation scale must be positive")

trans_ex2 = exaggerate_transforms(transforms, trans_scale=translation_scale, rot_scale=rotation_scale)
visualize_image_positions(trans_ex2, shapes, labels=labels, translate_scale=1.0, show_original=True, auto_zoom=True)

### 5.2 Visualisation de l'Alignement

Pour vérifier que l'alignement a bien fonctionné, on peut superposer les images et tracer des lignes entre les étoiles correspondantes.

- Les **points jaunes** sont les étoiles de l'image de référence.
- Les **points cyan** sont les étoiles de l'image à aligner, après transformation.
- Les **lignes vertes** connectent les paires d'étoiles qui ont été correctement associées (les "inliers").

Si l'alignement est réussi, les lignes vertes devraient toutes être horizontales.

In [ ]:
# @title
index_reference_image = 4 # @param {"type":"integer"}
index_matching_image = 1 # @param {"type":"integer"}

if index_reference_image < 0 or index_reference_image >= len(calibrated_frames):
    raise ValueError(f"Invalid reference image index. It should be between 0 and {len(calibrated_frames) - 1}")
if index_matching_image < 0 or index_matching_image >= len(calibrated_frames):
    raise ValueError(f"Invalid matching image index. It should be between 0 and {len(calibrated_frames) - 1}")
if index_reference_image == index_matching_image:
    raise ValueError("Reference and matching images cannot be the same")

temp = [percentile_normalize(calibrated_frames[i], 20, 99) for i in range(len(calibrated_frames))]
show_match_pair(temp, matches, transforms, index_matching_image, ref_idx=index_reference_image)

## 6. Dématriçage (Debayering)

Les capteurs couleur en astronomie utilisent une **matrice de Bayer** (souvent RGGB), où chaque pixel ne voit qu'une seule couleur (rouge, vert ou bleu). L'image brute est donc une mosaïque de pixels monochromes.

Le **dématriçage** est le processus qui reconstruit une image en couleur (RGB) à partir de cette mosaïque. Pour chaque pixel, l'algorithme interpole les deux couleurs manquantes en se basant sur les pixels voisins.

Nous appliquons cette étape après la calibration et avant l'alignement final pour obtenir des images en couleur.

<br>
<br>
<div>
<img src="https://images.nicolaspayot.ca/super-rez-workshop/pipeline_debayer.png" width="1200"/>
</div>
<br>
<br>

In [ ]:
pattern = "RGGB" #@param ['RGGB', 'GRBG', 'GBRG', 'BGGR'] {'type':'string'}
method = "VNG" #@param ['VNG', 'Bilinear', 'EA'] {'type':'string'}

def debayer_images(data, pattern, method):
    debayered = []
    for i in range(len(data)):
        temp = data[i]
        temp = (temp - temp.min()) / (temp.max() - temp.min()) * 255

        debayered.append(debayer_image(temp.astype(np.uint8), pattern=pattern, method=method))
    return np.array(debayered)


debayered_calibrated_frames = debayer_images(calibrated_frames, pattern, method)

In [ ]:
# @title
def update_stretch(val):
    stretch = ImageNormalize(stretch=AsinhStretch(a=val/1000))

    n_images = len(debayered_calibrated_frames)
    cols = 3
    rows = (n_images + cols - 1) // cols

    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).flatten()

    for i in range(n_images):
        ax = axes[i]
        image = crop(stretch(percentile_normalize(debayered_calibrated_frames[i], 60, 100)), size=800)
        ax.imshow(image - image.min())
        ax.set_title(f'Debayered image {i}')
        ax.axis('off')

    for i in range(n_images, len(axes)):
        axes[i].axis('off')

    plt.suptitle('Debayered Calibrated Frames', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

interact(update_stretch,
         val=FloatSlider(min=0.01, max=100, step=0.01, value=10,
                         description='Stretch (a*1000)', continuous_update=False))
None

## 7. Application des Transformations (Warping) et Empilement

Une fois les transformations calculées, on les applique à chaque image calibrée pour les aligner sur l'image de référence. C'est l'étape d'**alignement**.

Ensuite, on procède à l'**empilement** (ou "stacking") :
- On superpose toutes les images alignées.
- On calcule la valeur médiane ou la moyenne de chaque pixel.

L'empilement a deux avantages majeurs :
1.  **Réduction du bruit** : Le bruit aléatoire est lissé, ce qui donne une image beaucoup plus propre.
2.  **Augmentation du signal** : Le signal faible de la galaxie, présent dans chaque image, est renforcé.

Cela permet de révéler des détails et des extensions de la galaxie qui étaient invisibles dans les images individuelles.

<br>
<br>
<div>
<img src="https://images.nicolaspayot.ca/super-rez-workshop/pipeline_align_stack.png" width="1200"/>
</div>
<br>
<br>

In [ ]:
# Choose which reference index to warp into (0..N-1). Use the same index used when computing transforms if applicable.
REF_IDX = 0
interpolation = 'cubic' #@param {'type': 'string'}['cubic', 'nearest']

warped_debayered_calibrated_frames = warp_images_to_reference(debayered_calibrated_frames, transforms, ref_idx=REF_IDX,
                                                              interpolation=interpolation ,
                                                              border_value=0.0)

warped = np.array(warped_debayered_calibrated_frames) / 255

# Display large grid of warped images so small offsets are visible
n = len(warped)
cols = min(3, n)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 4 * rows))
axes = np.array(axes).reshape(-1)
for i, ax in enumerate(axes):
    if i >= n:
        ax.axis('off')
        continue
    wimg = warped[i]
    # if multichannel, show as RGB, otherwise grayscale
    if wimg is None:
        ax.set_title(f'img{i}: None')
        ax.axis('off')
        continue
    if wimg.ndim == 3 and wimg.shape[2] == 3:
        disp = percentile_normalize(wimg, 1, 99.9)
        ax.imshow(disp[..., ::])
    else:
        arr = np.asarray(wimg).astype(float)
        pmin, pmax = np.percentile(arr, (1, 99))
        ax.imshow(np.clip((arr - pmin) / (pmax - pmin + 1e-12), 0, 1), cmap='gray')
    ax.set_title(f'warped img {i}')
    ax.axis('off')
plt.suptitle(f'Warped images into reference {REF_IDX}')
plt.tight_layout()
plt.show()

## 8. Résultat Final

Finalement, nous pouvons faire une calibration des couleurs pour obtenir un arrière-plan noir neutre et des couleurs naturelles. On peut aussi utiliser les techniques d'intelligence artificielle vues précédemment pour améliorer encore la qualité de l'image.

<br>
<br>
<div>
<img src="https://images.nicolaspayot.ca/super-rez-workshop/pipeline_final.png" width="1200"/>
</div>
<br>
<br>


Après toutes ces étapes, nous obtenons enfin notre image finale !

Comparez cette image avec les images brutes du début. Vous remarquerez :
- Une **nette amélioration du rapport signal/bruit**.
- La **disparition des défauts** (pixels chauds, vignettage).
- Des **détails beaucoup plus fins** dans les bras spiraux et le bulbe de la galaxie.

Ce pipeline traditionnel (sans l'intelligence artificielle) est la base de l'astrophotographie du ciel profond et permet de transformer des données bruitées en une image de qualité.

In [ ]:
# @title
master_image = make_master_frame(warped_debayered_calibrated_frames)
master_image /= np.mean(master_image, axis=(0, 1))

In [ ]:
# @title
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np

def update_stretch_percentile(val, down, up, r, g, b):
    up = 100 - up / 1000
    if down >= up:
        down = 0
        up = 100
    stretch = ImageNormalize(stretch=AsinhStretch(a=val / 1000))
    image = stretch(percentile_normalize(master_image[..., :], down, up)) * np.array((r, g, b))

    fig = plt.figure(figsize=(12, 6), dpi=100, facecolor='none')

    gs = gridspec.GridSpec(2, 3, width_ratios=[1, 1, 0.8], height_ratios=[1, 1])

    # Image Plot
    ax_img = fig.add_subplot(gs[:, :2])
    ax_img.imshow(clip(image))
    ax_img.axis('off')

    # Histogram Plot
    ax_hist = fig.add_subplot(gs[0, 2])

    ax_hist.patch.set_alpha(0.0)

    colors = ['r', 'g', 'b']
    channel_names = ['Red', 'Green', 'Blue']
    for i, color in enumerate(colors):
        hist, bins = np.histogram(image[..., i].ravel(), bins=256, range=(0,1))
        ax_hist.plot(bins[:-1], hist, color=color, label=channel_names[i], alpha=0.7, linewidth=1.5)

    ax_hist.set_xlim([0, 1])
    ax_hist.set_title("Histogram", fontsize=10)

    ax_hist.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


interact(update_stretch_percentile,
         val=FloatSlider(min=0.01, max=100, step=0.01, value=10,
                         description='Stretch (a*1000)', continuous_update=False),
         down=FloatSlider(min=0.01, max=90, step=0.01, value=0,
                          description='Bottom Percentile', continuous_update=False),
         up=FloatSlider(min=0, max=100, step=0.01, value=0,
                        description='Upper Percentile (percentile * 100)', continuous_update=False),
         r=FloatSlider(min=0.1, max=2.0, step=0.01, value=1.0),
         g=FloatSlider(min=0.1, max=2.0, step=0.01, value=1.0),
         b=FloatSlider(min=0.1, max=2.0, step=0.01, value=1.0)
         )
None

## 9. Ajout de l'Intelligence Artificielle

Enfin, chargeons l'image obtenue avec l'addition de l'intelligence artificielle à ce pipeline. Malheureusement, nous ne pouvons pas faire fonctionner cette intelligence artificielle directement dans ce notebook à cause des limitations de ressources. Nous avons donc pré-calculé cette image pour vous.

In [ ]:
# @title
image_path = proj_root / 'data' / 'NGC7331.png'

img = mpimg.imread(str(image_path))

plt.figure(figsize=(12, 12))
plt.imshow(crop(img, size=3000))
plt.axis('off')
plt.show()

# Section suivante

Pour passer au « Notebook » suivant, cliquez sur <a href="https://colab.research.google.com/github/GabrielMissael/super-resolution-workshop/blob/master/notebooks/Diffusion_Simulated_Galaxy_Pipeline_fr_v2.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Ouvrir dans Colab"/></a>
